# Find Published ITSN1 Variants in Non-Cohort UKB
Author: Sheila Yeboah

Date Modified: 3/11/26

Description: Find ITSN1 variants that were previously published (Spargo et. al Skuladdotir et. al) but not found in our cohort

In [8]:
GENE = "ITSN1_WGS"
gene_names = ["ITSN1"]
OUTPUT_DIR = f"/results/Pull_Published_Variants"
DATA_DIR = "/opt/notebooks/data"
ancestry_list = ["AAC", "AFR", "AJ", "AMR", "CAH", "CAS", "EAS", "EUR", "FIN", "MDE", "SAS"]

## Imports and Notebook Setup

In [2]:
## Import the necessary packages 
import os
import numpy as np
import pandas as pd
import math
import sys
import requests
import subprocess
#import statsmodels.api as sm
import scipy
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from datetime import date, datetime
import pathlib
#from plotnine import *
import pyspark
import dxdata
import dxpy


## Print out package versions
## Getting packages loaded into this notebook and their versions to allow for reproducibility
    # Repurposed code from stackoverflow here: https://stackoverflow.com/questions/40428931/package-for-listing-version-of-packages-used-in-a-jupyter-notebook

## Import packages 
import pkg_resources
import types
from datetime import date
today = date.today()
date = today.strftime("%d-%b-%Y").upper()

## Define function 
def get_imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            # Split ensures you get root package, not just imported function
            name = val.__name__.split(".")[0]

        elif isinstance(val, type):
            name = val.__module__.split(".")[0]

        # Some packages are weird and have different imported names vs. system/pip names
        # Unfortunately, there is no systematic way to get pip names from a package's imported name. You'll have to add exceptions to this list manually!
        poorly_named_packages = {
            "PIL": "Pillow",
            "sklearn": "scikit-learn"
        }
        if name in poorly_named_packages.keys():
            name = poorly_named_packages[name]

        yield name

## Get a list of packages imported 
imports = list(set(get_imports()))

# The only way I found to get the version of the root package from only the name of the package is to cross-check the names of installed packages vs. imported packages
requirements = []
for m in pkg_resources.working_set:
    if m.project_name in imports and m.project_name!="pip":
        requirements.append((m.project_name, m.version))

## Print out packages and versions 
print(f"PACKAGE VERSIONS ({date})")
for r in requirements:
    print("\t{}=={}".format(*r))

PACKAGE VERSIONS (09-JAN-2026)
	dxdata==0.47.0
	dxpy==0.396.0
	matplotlib==3.9.2
	numpy==1.26.4
	pandas==2.2.3
	pyspark==3.5.2
	requests==2.32.3
	scipy==1.15.3
	seaborn==0.12.2


/tmp/ipykernel_93/1700372142.py:28: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [ ]:
# define helper function
def fetch_gene_info_ensembl(gene_names, species="human", genome_version="GRCh38"):
    gene_info_dict = {}
    server = "https://rest.ensembl.org"
    
    for gene_name in gene_names:
        endpoint = f"/lookup/symbol/{species}/{gene_name}"
        headers = {"Content-Type": "application/json"}

        response = requests.get(server + endpoint, headers=headers, params={"expand": "1"})
        if not response.ok:
            print(f"Fetching failed for {gene_name}")
            continue

        data = response.json()
        gene_info = {
            "gene_name": data.get("display_name", gene_name),
            "chromosome": f"chr{data['seq_region_name']}",
            "start": int(data["start"]),
            "end": int(data["end"]),
            "genome_version": genome_version
        }

        gene_info_dict[gene_name] = gene_info

    return gene_info_dict

In [ ]:
# initialize Spark
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

## Pull UKB Participants that are not part of analysis cohort

In [19]:
# pull participant information using Spark
dispensed_dataset_id = dxpy.find_one_data_object(typename="Dataset", name="app*.dataset", folder="/", name_mode="glob")["id"]
dataset = dxdata.load_dataset(id=dispensed_dataset_id)
participant = dataset["participant"]

In [ ]:
# filter by appropriate fields

field_names = [
    "eid", 
    "p31", 
    "p34", 
    "p21022", 
    "p40000_i0",
]
df_participants = participant.retrieve_fields(names=field_names, coding_values="replace", engine=dxdata.connect())
df_participants = df_participants.toPandas()

In [ ]:
# change columns to readable names
df_participants = df_participants.rename(columns={
    "eid":"ID",
    "p31":"GENETIC_SEX", 
    "p34":"BIRTH_YEAR", 
    "p21022":"AGE_OF_RECRUIT",
    "p42032":"PD_DATE",
    "p40000_i0":"DATE_OF_DEATH",
})

In [26]:
# convert dnanexus ids to numeric
df_participants["ID"] = pd.to_numeric(df_participants["ID"])

In [ ]:
# pull cohort sample ids from project folder
! dx download {OUTPUT_DIR}/sample_ids.txt

In [ ]:
# read in ids
samples_tested = pd.read_csv("sample_ids.txt", sep="\t", header=None)
samples_tested

In [ ]:
# remove samples that we already tested from list of DNA nexus participant ids

df_participants = df_participants[~(df_participants["ID"].isin(samples_tested[0]))]

In [ ]:
# write these non-cohort ids to file
df_participants["ID"].to_csv("samples_ids_not_tested.txt", sep="\t", index=False, header=None)

In [ ]:
# upload to DNAnexus 
! dx upload samples_ids_not_tested.txt --path $OUTPUT_DIR/

In [ ]:
# get ITSN1 bvals using helper function, to pull vcfs
gene_info = fetch_gene_info_ensembl(gene_names)
print(gene_info)
for gene_name in gene_names:
    start = gene_info[gene_name]["start"] // 20000
    end = fetch_gene_info_ensembl(gene_names)[gene_name]["end"] // 20000 + 1
    print(f"Chromosome:    {fetch_gene_info_ensembl(gene_names)[gene_name]['chromosome']}")
    print(f"Start b-val:   {start}")
    print(f"End b-val:     {end}")

{'ITSN1': {'gene_name': 'ITSN1', 'chromosome': 'chr21', 'start': 33642400, 'end': 33899861, 'genome_version': 'GRCh38'}}
Chromosome:    chr21
Start b-val:   1682
End b-val:     1695


## Get Non-Cohort Participant ITSN1 region genotypes

### Pull pvcf chunks

In [ ]:
%%bash -s $OUTPUT_DIR

# Pull vcf files from bulk data
for b_val in {1682..1695};
do
    dx run swiss-army-knife \
    -iin="/Bulk/DRAGEN\ WGS/DRAGEN\ population\ level\ WGS\ variants,\ pVCF\ format\ [500k\ release]/chr21/ukb24310_c21_b${b_val}_v1.vcf.gz" \
    -iin="/Bulk/DRAGEN\ WGS/DRAGEN\ population\ level\ WGS\ variants,\ pVCF\ format\ [500k\ release]/chr21/ukb24310_c21_b${b_val}_v1.vcf.gz.tbi" \
    -iin="${1}/samples_ids_not_tested.txt" \
    -icmd="bcftools view -O z -S samples_ids_not_tested.txt --force-samples ukb24310_c21_b${b_val}_v1.vcf.gz -o ITSN1_b${b_val}_age60cutoff.vcf.gz" \
    --instance-type mem2_ssd1_v2_x32 \
    --brief \
    --destination "${projectid}:${1}/01_pvcf_chunks/"
done

### Merge pvcf chunks

In [ ]:
%%bash -s $OUTPUT_DIR

# merge pvcfs into one file
dx run swiss-army-knife \
-iin="${1}/01_pvcf_chunks/ITSN1_b1682_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1683_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1684_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1685_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1686_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1687_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1688_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1689_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1690_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1691_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1692_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1693_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1694_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1695_age60cutoff.vcf.gz" \
-icmd="bcftools concat -O z ITSN1_b1682_age60cutoff.vcf.gz ITSN1_b1683_age60cutoff.vcf.gz ITSN1_b1684_age60cutoff.vcf.gz ITSN1_b1685_age60cutoff.vcf.gz ITSN1_b1686_age60cutoff.vcf.gz ITSN1_b1687_age60cutoff.vcf.gz ITSN1_b1688_age60cutoff.vcf.gz ITSN1_b1689_age60cutoff.vcf.gz ITSN1_b1690_age60cutoff.vcf.gz ITSN1_b1691_age60cutoff.vcf.gz ITSN1_b1692_age60cutoff.vcf.gz ITSN1_b1693_age60cutoff.vcf.gz ITSN1_b1694_age60cutoff.vcf.gz ITSN1_b1695_age60cutoff.vcf.gz -o ITSN1_age60cutoff.vcf.gz" \
--instance-type mem2_ssd1_v2_x64 \
--name merge_pvcfs \
--brief \
--destination "${projectid}:${1}/02_pvcf_genes"

job-J5Kq8BQJJFG2P71q4kKvfFz1


### Split multiallelic and normalize pvcf

In [2]:
%%bash -s $OUTPUT_DIR

dx run swiss-army-knife \
-iin="${1}/02_pvcf_genes/ITSN1_age60cutoff.vcf.gz" \
-icmd="bcftools norm -m-both -o biallelic_age60cutoff.vcf ITSN1_age60cutoff.vcf.gz" \
--instance-type mem2_ssd1_v2_x64 \
--brief \
--destination "${projectid}:${1}/03_normalized"

job-J5PF08QJJFG98XVfKBYGpbP5


In [3]:
%%bash -s $OUTPUT_DIR

dx run swiss-army-knife \
-iin="${1}/03_normalized/biallelic_age60cutoff.vcf" \
-iin="/data/hg38.fa" \
-icmd="bcftools norm -f hg38.fa -o normalized_age60cutoff.vcf biallelic_age60cutoff.vcf" \
--brief \
--instance-type mem2_ssd1_v2_x64 \
--name "normalize pvcf" \
--destination "${projectid}:${1}/03_normalized/"

job-J5PKFGQJJFG16b3z9J1f2vp6


In [ ]:
%%bash -s $OUTPUT_DIR

### remove sites with MAC <1

dx run swiss-army-knife \
-iin="${1}/03_normalized/normalized_age60cutoff.vcf" \
-iin="/data/hg38.fa" \
-icmd="bcftools view -i 'MAC>=1' normalized_age60cutoff.vcf -O v -o normalized_mac1_age60cutoff.vcf" \
--brief \
--instance-type mem2_ssd1_v2_x64 \
--destination "${projectid}:${1}/03_normalized/"

### Update variant ids to chr:pos:r:a format

In [ ]:
import subprocess
# Create file with updated variant ids in chr:r:a format
projectid = "project"
plink_cmd = (
    f"plink2 --vcf normalized_age60cutoff.vcf "
    f"--set-all-var-ids 'chr@:#:\$r:\$a' "
    f"--new-id-max-allele-len 999 "
    f"--double-id "
    f"--mac 1 "
    f"--make-pgen "
    f"--pheno-name PHENO "
    f"--out normalized_mac1_age60cutoff_updateids"
)

# Construct the full dx run command
dx_cmd = (
    f'dx run swiss-army-knife '
    f'-iin="{OUTPUT_DIR}/03_normalized/normalized_age60cutoff.vcf" '
    f'-icmd="{plink_cmd}" '
    f'--brief '
    f'--instance-type mem2_ssd1_v2_x64 '
    f'--destination "{projectid}:{OUTPUT_DIR}/03_normalized/"'
)

# Print for verification
print("Running command:\n", dx_cmd)

# Run the command
subprocess.run(dx_cmd, shell=True, check=True)

### Filter to published variants

In [ ]:
# download prev published variants

! dx download {OUTPUT_DIR}/all_prev_published_variants.tsv

In [ ]:
# take the variant id column and create a new file
! cut -f 1 all_prev_published_variants.tsv | tail -n +2 > previously_published_variant_ids.tsv

In [ ]:
! dx upload previously_published_variant_ids.tsv --path {OUTPUT_DIR}/

In [ ]:
# Extract previously published variants
plink_cmd = (
    f"plink2 "
    f"--pfile normalized_mac1_age60cutoff_updateids "
    f"--extract previously_published_variant_ids.tsv "
    f"--make-pgen erase-dosage "
    f"--mac 1 "
    f"--out previously_published_variant"
)

# Build dx run command
cmd = [
    "dx", "run", "swiss-army-knife",
    f"-iin={OUTPUT_DIR}/03_normalized/normalized_mac1_age60cutoff_updateids.pgen",
    f"-iin={OUTPUT_DIR}/03_normalized/normalized_mac1_age60cutoff_updateids.psam",
    f"-iin={OUTPUT_DIR}/03_normalized/normalized_mac1_age60cutoff_updateids.pvar",
    f"-iin={OUTPUT_DIR}/previously_published_variant_ids.tsv",
    f"-icmd={plink_cmd}",
    "--instance-type", "mem2_ssd1_v2_x64",
    "--brief",
    "--name", "extract published variants",
    "--destination", f"{projectid}:{OUTPUT_DIR}/"
]

subprocess.run(cmd)

### Generate recode file

In [ ]:
# get recode file to do carrier analysis

plink_cmd = (
    f"plink2 "
    f"--pfile previously_published_variant "
    f"--recode A "
    f"--out previously_published_variant"
)

    # dx run swiss-army-knife wrapper
cmd = [
    # PLINK bfile inputs 
   "dx", "run", "swiss-army-knife",
    f"-iin={OUTPUT_DIR}/previously_published_variant.pgen",
    f"-iin={OUTPUT_DIR}/previously_published_variant.psam",
    f"-iin={OUTPUT_DIR}/previously_published_variant.pvar",
    f"-icmd={plink_cmd}",
    "--instance-type", "mem2_ssd1_v2_x64",
    "--brief",
    "--name", "PLINK recode",
    "--destination", f"{projectid}:{OUTPUT_DIR}/"
]

subprocess.run(cmd)

In [ ]:
! dx download {OUTPUT_DIR}/previously_published_variant.raw

## Carrier Investigation

### Get ITSN1 variant carriers from recode file

In [5]:
# read in .raw file

recode = pd.read_csv("previously_published_variant.raw", sep="\t")

In [ ]:
# get rows where carriers of at least one variant of interest are
meta_cols = recode.columns[:6]
geno_cols = recode.columns[6:]

carriers = recode[(recode[geno_cols] < 2).any(axis=1)].drop(columns=["PAT", "MAT", "SEX", "PHENOTYPE"])
carriers

In [ ]:
# write carriers to file
carriers.to_csv("ukb_carriers_of_prev_published_variants_not_in_analysis.csv", index=False)

In [ ]:
# upload to dnanexus
! dx upload ukb_carriers_of_prev_published_variants_not_in_analysis.csv --path {OUTPUT_DIR}/

### Focused investigation on published variants we did not find in original analysis

In [20]:
%%bash
# previously published ukb variants that are missing from original analysis
# Create a file with the SNPs that are in the pvar
cat > snplist.txt << EOF
chr21:33722604:C:CT
chr21:33722614:CA:C
chr21:33750145:A:G
chr21:33750175:AC:A
chr21:33750252:A:G
chr21:33750261:GC:G
chr21:33750267:C:CA
chr21:33755348:C:A
chr21:33761986:G:A
chr21:33765890:TG:T
chr21:33772120:C:T
chr21:33772153:G:T
chr21:33772324:G:A
chr21:33774727:A:C
chr21:33774756:C:T
chr21:33774809:CAAAG:C
chr21:33774853:CTT:C
chr21:33781460:G:A
chr21:33782101:C:T
chr21:33782134:G:GTAAC
chr21:33794346:AAG:A
chr21:33794425:C:T
chr21:33797380:C:T
chr21:33799930:G:C
chr21:33810975:G:GTGGA
chr21:33811111:C:G
chr21:33811223:G:A
chr21:33811223:G:T
chr21:33818288:C:T
chr21:33818294:C:T
chr21:33818380:CAT:C
chr21:33818434:C:G
chr21:33819281:C:T
chr21:33834414:ATT:A
chr21:33836440:G:C
chr21:33836476:C:T
chr21:33836582:T:TC
chr21:33836586:A:ACTTG
chr21:33836589:CA:C
chr21:33856786:C:T
chr21:33865182:G:T
chr21:33867313:C:G
chr21:33882259:C:A
chr21:33885049:G:A
chr21:33886397:C:T
EOF


In [ ]:
! dx download {OUTPUT_DIR}/all_prev_published_variants.tsv

In [ ]:
# read in previously published variants
prev_published_df = pd.read_csv("all_prev_published_variants.tsv", sep="\t")
prev_published_df

,SNP,Paper.Annotation,VEP.Pick.Allele.Consequence,Paper,Originally.Found.In,GP2.R11.IMP,GP2.R11.WGS,UKB.WGS,AoU.WGS
0,chr21:33811223:G:A,splice donor variant,splice_donor_variant,NaturePD,UKB,.,.,y,y
1,chr21:33772120:C:T,stop_gained,stop_gained,Cell,"UKB, AOU",.,.,y,y
2,chr21:33750145:A:G,missense_variant&splice_region_variant,"missense_variant,splice_region_variant",Cell,AOU,.,y,y,significant
3,chr21:33818473:G:A,"splice_donor_variant&intron_variant, splice donor",.,"Cell, NaturePD",AMP_PD,.,y,.,significant
4,chr21:33886397:C:T,stop_gained,.,"NaturePD, NaturePD","Iceland (deCODE genetics), UKB",.,y,.,significant
...,...,...,...,...,...,...,...,...,...
86,chr21:33882259:C:A,stop_gained,.,NaturePD,UKB,.,.,.,.
87,chr21:33885049:G:A,stop_gained,.,"Cell, NaturePD",UKB,.,.,.,.
88,chr21:33794425:C:T,stop_gained,.,"NaturePD, NaturePD","UKB, AMP_PD",.,y,.,.
89,chr21:33797380:C:T,"stop_gained&splice_region_variant, stop_gained",.,"Cell, NaturePD","UKB, AOU, UKB",.,.,.,.


In [ ]:
# make column for variants found in UKB participants that were not in the cohort
snps = pd.read_csv("snplist.txt", sep="\t", header=None, names=["SNP"])
snps["Found in non-cohort UKB"] = "y"
snps_df  = snps.copy()
snps_df

,SNP,Found in non-cohort UKB
0,chr21:33722604:C:CT,y
1,chr21:33722614:CA:C,y
2,chr21:33750145:A:G,y
3,chr21:33750175:AC:A,y
4,chr21:33750252:A:G,y
5,chr21:33750261:GC:G,y
6,chr21:33750267:C:CA,y
7,chr21:33755348:C:A,y
8,chr21:33761986:G:A,y
9,chr21:33765890:TG:T,y


In [32]:
# merge them on snp id
variants_in_noncohort_ukb = pd.merge(prev_published_df, snps_df, on="SNP", how="left")

In [ ]:
# write prev published snps to file
variants_in_noncohort_ukb.to_csv("all_prev_published_variants_with_noncohort_ukb_info.txt", sep="\t", index=False, na_rep="na")

In [ ]:
# upload files to dnanexus folder

! dx upload "all_prev_published_variants_with_noncohort_ukb_info.txt" --path {OUTPUT_DIR}/
! dx upload "snplist.txt" --path {OUTPUT_DIR}/
! dx upload "ukb_carriers_of_prev_published_variants_not_in_analysis.csv" --path {OUTPUT_DIR}/